# 实验七：头一转，画面为什么跟不上？
## 360° / VR Viewport Streaming、Head Motion 与感知 QoE

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~15 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **GPU 非必需** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **360° 内容覆盖整个球面，但用户每一刻只看其中一小块。网络能不能在用户转头之前，把“正确位置”的高质量内容送到？**

这是整个 FMI Hands-on Lab 的收官实验。


## Demo6 → Demo7：从“播放器是否流畅”到“用户下一秒看哪里”

Demo6 关注：

**Network → Buffer → Playback QoE**

Demo7 新增了沉浸式媒体独有的问题：

**Head Motion → Viewport → Tile Selection → Network Delay → Viewport Mismatch**

因此，360° / VR Streaming 不只是“更高分辨率的视频”，而是一个**随用户视角动态变化的空间传输问题**。


## 学习目标

完成本实验后，你应该能够：

1. 正确解释 ERP（Equirectangular Projection）与球面经纬度的关系；
2. 理解头显看到的是 Perspective Viewport，而不是 ERP 上简单裁出的矩形；
3. 理解为什么 Viewport-Dependent Streaming 能节省大量带宽；
4. 理解快速 Head Turn + 网络延迟如何造成 **Viewport Mismatch**；
5. 使用 **Viewport Hit Ratio** 衡量当前视野中高质量内容是否及时到达；
6. 分析 Guard Band 如何在 **Bandwidth 与 Robustness** 之间取舍；
7. 理解 Head Pose Prediction 为什么能提升未来 Viewport 命中率；
8. 区分 Viewport Streaming 与 Foveated Streaming；
9. 将 Demo1–Demo7 串成完整的 Future Media & Internet 技术链。


## 核心概念

### 1. ERP：把球面展开成 2:1 矩形

ERP 使用近似线性的经纬度映射：

\[
x \propto longitude,\qquad y \propto latitude
\]

它不是 Mercator Projection。

ERP 的典型问题是：靠近南北极存在明显 oversampling / geometric distortion，因此平面像素面积并不等价于球面面积。

### 2. Viewport

用户并不直接观看整张 ERP，而是根据当前：

- Yaw
- Pitch
- Field of View

从球面中投影出一个 Perspective View。

### 3. Viewport-Dependent Streaming

把 ERP 划分成 Tiles：

- 当前 Viewport：High Quality
- 周围 Guard Band：High / Medium Quality
- 其他区域：Low Quality

目标是：

> **不要把用户暂时看不到的整个球面都以最高质量传输。**


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 计算资源 | CPU |
| GPU | 不需要；可作为 Head Pose Predictor 扩展 |
| Internet | Off |
| ERP | 960×480 |
| Viewport | 320×240 |
| Head Trace | 6 秒，10 Hz |
| Tile Grid | 8 × 4 |
| Network Latency | 100 / 200 / 300 ms 对比 |
| 依赖 | NumPy、Pandas、Matplotlib、Pillow、SciPy、scikit-image |

直接 **Run All** 即可。


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display, Image as IPImage

from scipy.ndimage import map_coordinates
from skimage import data, img_as_float32
from skimage.transform import resize

SEED = 2026
rng = np.random.default_rng(SEED)

ERP_H, ERP_W = 480, 960
VIEW_H, VIEW_W = 240, 320

FPS = 10
DURATION = 6
N_FRAMES = FPS * DURATION
TIMES = np.arange(N_FRAMES) / FPS

FOV_H = 100.0

TILE_ROWS = 4
TILE_COLS = 8
HQ_TILE_MBPS = 1.00
LQ_TILE_MBPS = 0.15

print("Environment ready")
print(f"ERP: {ERP_W}x{ERP_H}")
print(f"Viewport: {VIEW_W}x{VIEW_H} | horizontal FOV={FOV_H:.0f} deg")
print(f"Head trace: {DURATION}s @ {FPS} Hz | Tile grid={TILE_COLS}x{TILE_ROWS}")


# 第一幕：生成一张真正“方向可辨”的 ERP

为了让转头效果清楚，我们把四个离线自然场景放到四个主要方向：

- **Front**
- **Right**
- **Back**
- **Left**

横轴对应 Yaw：

\[
-180^\circ \rightarrow +180^\circ
\]

纵轴对应 Pitch：

\[
+90^\circ \rightarrow -90^\circ
\]

这不是一个真实相机拍摄的 360° panorama，但它能非常清楚地展示 ERP、方向、Viewport 和 Tile Streaming 的关系。


In [2]:
sources = [
    img_as_float32(data.astronaut()),
    img_as_float32(data.rocket()),
    img_as_float32(data.coffee()),
    img_as_float32(data.chelsea()),
]

labels = ["Left", "Front", "Right", "Back"]

panels = []
for source in sources:
    panel = resize(
        source,
        (ERP_H, ERP_W // 4),
        anti_aliasing=True,
        preserve_range=True,
    ).astype(np.float32)
    panels.append(panel)

erp = np.concatenate(panels, axis=1)

pil = Image.fromarray(np.uint8(np.clip(erp * 255, 0, 255)))
draw = ImageDraw.Draw(pil)

quarter_centers = [ERP_W//8, 3*ERP_W//8, 5*ERP_W//8, 7*ERP_W//8]
for x, label in zip(quarter_centers, labels):
    draw.rectangle((x-38, 8, x+38, 30), fill=(0, 0, 0))
    draw.text((x-30, 12), label, fill=(255, 255, 255))

draw.line((0, ERP_H//2, ERP_W, ERP_H//2), fill=(255,255,255), width=2)
erp = np.asarray(pil).astype(np.float32) / 255.0

plt.figure(figsize=(14, 7))
plt.imshow(erp)
plt.title("Synthetic Equirectangular Panorama (ERP)")
plt.xlabel("Yaw: -180°  ←  0°  →  +180°")
plt.ylabel("Pitch: +90° → -90°")
plt.xticks([0, ERP_W/4, ERP_W/2, 3*ERP_W/4, ERP_W-1],
           ["-180°", "-90°", "0°", "+90°", "+180°"])
plt.yticks([0, ERP_H/2, ERP_H-1], ["+90°", "0°", "-90°"])
plt.show()


# 第二幕：真正做 Perspective Viewport Extraction

头显并不是在 ERP 上简单裁一个矩形。

我们从 Viewport 中每个像素发出一条 3D ray，再通过：

- Yaw rotation
- Pitch rotation
- Sphere longitude / latitude
- ERP sampling

得到用户头部朝向对应的 Perspective View。

下面分别看：

- Yaw = 0°
- Yaw = +90°
- Yaw = 180°
- Yaw = -90°


In [3]:
def extract_viewport(
    panorama,
    yaw_deg=0.0,
    pitch_deg=0.0,
    fov_h_deg=FOV_H,
    out_h=VIEW_H,
    out_w=VIEW_W,
):
    aspect = out_w / out_h
    fov_h = np.deg2rad(fov_h_deg)
    fov_v = 2 * np.arctan(np.tan(fov_h / 2) / aspect)

    xs = np.linspace(-np.tan(fov_h/2), np.tan(fov_h/2), out_w)
    ys = np.linspace(np.tan(fov_v/2), -np.tan(fov_v/2), out_h)
    xx, yy = np.meshgrid(xs, ys)
    zz = np.ones_like(xx)

    norm = np.sqrt(xx**2 + yy**2 + zz**2)
    x = xx / norm
    y = yy / norm
    z = zz / norm

    pitch = np.deg2rad(pitch_deg)
    y2 = y*np.cos(pitch) - z*np.sin(pitch)
    z2 = y*np.sin(pitch) + z*np.cos(pitch)
    x2 = x

    yaw = np.deg2rad(yaw_deg)
    x3 = x2*np.cos(yaw) + z2*np.sin(yaw)
    z3 = -x2*np.sin(yaw) + z2*np.cos(yaw)
    y3 = y2

    longitude = np.arctan2(x3, z3)
    latitude = np.arcsin(np.clip(y3, -1, 1))

    map_x = ((longitude + np.pi) / (2*np.pi) * panorama.shape[1]) % panorama.shape[1]
    map_y = (np.pi/2 - latitude) / np.pi * panorama.shape[0]
    map_y = np.clip(map_y, 0, panorama.shape[0]-1)

    output = np.empty((out_h, out_w, 3), dtype=np.float32)
    for channel in range(3):
        output[..., channel] = map_coordinates(
            panorama[..., channel],
            [map_y, map_x],
            order=1,
            mode="nearest",
        )

    return np.clip(output, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, yaw in zip(axes, [0, 90, 180, -90]):
    view = extract_viewport(erp, yaw_deg=yaw)
    ax.imshow(view)
    ax.set_title(f"Yaw = {yaw}°")
    ax.axis("off")

plt.suptitle("Perspective Viewports from the Same ERP")
plt.tight_layout()
plt.show()


# 第三幕：Head Motion Timeline

现在加入真正的用户行为。

6 秒轨迹：

- 0–2.0 s：看正前方
- 2.0–2.5 s：快速向右转到 +150°
- 2.5–4.3 s：保持
- 4.3–4.9 s：快速回转到 -60°
- 4.9–6.0 s：保持

快速转头时，网络最容易出现：

> **“高质量区域还停留在旧 Viewport，而用户已经看向新方向。”**


In [4]:
yaw_trace = np.zeros(N_FRAMES, dtype=np.float32)
pitch_trace = np.zeros(N_FRAMES, dtype=np.float32)

for i, t in enumerate(TIMES):
    if t < 2.0:
        yaw_trace[i] = 0
    elif t < 2.5:
        yaw_trace[i] = 150 * (t - 2.0) / 0.5
    elif t < 4.3:
        yaw_trace[i] = 150
    elif t < 4.9:
        yaw_trace[i] = 150 - 210 * (t - 4.3) / 0.6
    else:
        yaw_trace[i] = -60

plt.figure(figsize=(11, 4))
plt.plot(TIMES, yaw_trace, marker="o")
plt.xlabel("Time (s)")
plt.ylabel("Head Yaw (deg)")
plt.title("6-second Head Motion Trace")
plt.grid(alpha=0.2)
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, idx in zip(axes, [10, 22, 30, 47]):
    ax.imshow(extract_viewport(erp, yaw_trace[idx], pitch_trace[idx]))
    ax.set_title(f"t={TIMES[idx]:.1f}s | yaw={yaw_trace[idx]:.0f}°")
    ax.axis("off")
plt.suptitle("The User's View Changes with Head Motion")
plt.tight_layout()
plt.show()


# 第四幕：把 ERP 划成 Tiles

将全景划分为 **8 × 4 = 32 个 tiles**。

教学版码率模型：

- High-quality tile：1.00 Mbps
- Low-quality tile：0.15 Mbps

三种基本策略：

### Full Sphere HQ
全部 32 个 tiles 最高质量。

### Viewport Only
只有当前预测 Viewport 对应 tiles 最高质量。

### Guard Band
Viewport 周围额外传一些高质量 tiles，提高突然转头时的容错。

这一步先建立“空间不均匀分配码率”的直觉。


In [5]:
tile_w_deg = 360 / TILE_COLS
tile_h_deg = 180 / TILE_ROWS

tile_yaws = -180 + (np.arange(TILE_COLS) + 0.5) * tile_w_deg
tile_pitches = 90 - (np.arange(TILE_ROWS) + 0.5) * tile_h_deg

def angular_diff(a, b):
    return (a - b + 180) % 360 - 180

def tiles_for_pose(yaw_deg, pitch_deg=0.0, guard_deg=0.0):
    selected = set()

    half_h = FOV_H / 2 + guard_deg
    # Vertical FOV is derived from aspect ratio.
    fov_v = np.rad2deg(
        2 * np.arctan(
            np.tan(np.deg2rad(FOV_H)/2) / (VIEW_W/VIEW_H)
        )
    )
    half_v = fov_v / 2 + guard_deg

    for r, tile_pitch in enumerate(tile_pitches):
        for c, tile_yaw in enumerate(tile_yaws):
            if (
                abs(angular_diff(tile_yaw, yaw_deg)) < half_h + tile_w_deg/2
                and abs(tile_pitch - pitch_deg) < half_v + tile_h_deg/2
            ):
                selected.add((r, c))

    return selected

def draw_tile_grid(panorama, selected=None, title=""):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(panorama)

    for c in range(1, TILE_COLS):
        ax.axvline(c * ERP_W / TILE_COLS, linewidth=1)
    for r in range(1, TILE_ROWS):
        ax.axhline(r * ERP_H / TILE_ROWS, linewidth=1)

    if selected is not None:
        for r, c in selected:
            x0 = c * ERP_W / TILE_COLS
            y0 = r * ERP_H / TILE_ROWS
            rect = plt.Rectangle(
                (x0, y0),
                ERP_W / TILE_COLS,
                ERP_H / TILE_ROWS,
                fill=False,
                linewidth=3,
            )
            ax.add_patch(rect)

    ax.set_title(title)
    ax.axis("off")
    plt.show()

example_tiles = tiles_for_pose(0, 0, guard_deg=0)
draw_tile_grid(
    erp,
    example_tiles,
    "8×4 Tile Grid | Highlighted = Current Viewport Tiles"
)

print(f"Current viewport uses {len(example_tiles)} of {TILE_ROWS*TILE_COLS} tiles")


# 第五幕：Full Sphere HQ vs Viewport Streaming

如果全部 32 tiles 都用最高质量：

> 带宽最高，但随便转头都有高质量。

如果只把当前 Viewport 对应 tiles 设为 HQ：

> 带宽显著下降，但依赖及时、准确的 Head Pose。

下面先忽略网络延迟，比较理论带宽。


In [6]:
def bandwidth_for_hq_tiles(n_hq):
    n_total = TILE_ROWS * TILE_COLS
    return n_hq * HQ_TILE_MBPS + (n_total - n_hq) * LQ_TILE_MBPS

full_hq_tiles = TILE_ROWS * TILE_COLS

viewport_counts = [
    len(tiles_for_pose(y, p, 0))
    for y, p in zip(yaw_trace, pitch_trace)
]

guard20_counts = [
    len(tiles_for_pose(y, p, 20))
    for y, p in zip(yaw_trace, pitch_trace)
]

bandwidth_table = pd.DataFrame([
    {
        "Strategy": "Full Sphere HQ",
        "Avg HQ Tiles": full_hq_tiles,
        "Avg Bandwidth (Mbps)": bandwidth_for_hq_tiles(full_hq_tiles),
    },
    {
        "Strategy": "Viewport Only",
        "Avg HQ Tiles": np.mean(viewport_counts),
        "Avg Bandwidth (Mbps)": np.mean([bandwidth_for_hq_tiles(n) for n in viewport_counts]),
    },
    {
        "Strategy": "Viewport + 20° Guard",
        "Avg HQ Tiles": np.mean(guard20_counts),
        "Avg Bandwidth (Mbps)": np.mean([bandwidth_for_hq_tiles(n) for n in guard20_counts]),
    },
])

display(bandwidth_table.style.format({
    "Avg HQ Tiles": "{:.1f}",
    "Avg Bandwidth (Mbps)": "{:.1f}",
}))


# 第六幕：Latency Challenge —— 高质量区域为什么会“跟不上头”？

Viewport Streaming 依赖用户 Head Pose。

如果端到端控制 / 请求 / 内容到达延迟是：

- 100 ms
- 200 ms
- 300 ms

服务器看到的实际上是**过去的头部方向**。

快速转头时：

**Current Viewport ≠ Delivered HQ Viewport**

定义：

\[
Viewport\ Hit\ Ratio =
\frac{\text{当前 Viewport 中已是 HQ 的 tiles}}
{\text{当前 Viewport 所需 tiles}}
\]

100% 表示当前视野全部由 HQ tiles 覆盖。


In [7]:
def delayed_pose(index, latency_ms):
    lag = int(round(latency_ms / 1000 * FPS))
    return max(0, index - lag)

def hit_ratio(current_tiles, delivered_tiles):
    if not current_tiles:
        return 1.0
    return len(current_tiles & delivered_tiles) / len(current_tiles)

latency_rows = []
latency_curves = {}

for latency_ms in [100, 200, 300]:
    ratios = []

    for i in range(N_FRAMES):
        current = tiles_for_pose(yaw_trace[i], pitch_trace[i], 0)
        j = delayed_pose(i, latency_ms)
        delivered = tiles_for_pose(yaw_trace[j], pitch_trace[j], 0)
        ratios.append(hit_ratio(current, delivered))

    latency_curves[latency_ms] = np.array(ratios)

    latency_rows.append({
        "Latency (ms)": latency_ms,
        "Mean Viewport Hit": np.mean(ratios),
        "Minimum Viewport Hit": np.min(ratios),
    })

latency_table = pd.DataFrame(latency_rows)

display(latency_table.style.format({
    "Mean Viewport Hit": "{:.1%}",
    "Minimum Viewport Hit": "{:.1%}",
}))

plt.figure(figsize=(11, 5))
for latency_ms, ratios in latency_curves.items():
    plt.plot(TIMES, ratios * 100, marker="o", label=f"{latency_ms} ms")
plt.xlabel("Time (s)")
plt.ylabel("Viewport Hit Ratio (%)")
plt.title("Network Latency Creates Viewport Mismatch During Fast Head Turns")
plt.ylim(0, 105)
plt.grid(alpha=0.2)
plt.legend()
plt.show()


# 第七幕：Guard Band —— 花更多带宽买“转头保险”

如果只传当前 Viewport：

> 带宽最低，但高速转头时容易 miss。

可以在 Viewport 周围额外增加 Guard Band：

- 0°
- 20°
- 40°

Guard 越大：

- HQ tiles 越多
- Bandwidth 越高
- 快速转头时更鲁棒

这里固定 **200 ms latency**。


In [8]:
LATENCY_MS = 200
guard_rows = []
guard_curves = {}

for guard in [0, 20, 40]:
    ratios = []
    bandwidths = []

    for i in range(N_FRAMES):
        current = tiles_for_pose(yaw_trace[i], pitch_trace[i], 0)
        j = delayed_pose(i, LATENCY_MS)

        delivered = tiles_for_pose(
            yaw_trace[j],
            pitch_trace[j],
            guard
        )

        ratios.append(hit_ratio(current, delivered))
        bandwidths.append(bandwidth_for_hq_tiles(len(delivered)))

    guard_curves[guard] = np.array(ratios)

    guard_rows.append({
        "Guard Band": f"{guard}°",
        "Mean Viewport Hit": np.mean(ratios),
        "Minimum Viewport Hit": np.min(ratios),
        "Avg Bandwidth (Mbps)": np.mean(bandwidths),
    })

guard_table = pd.DataFrame(guard_rows)

display(guard_table.style.format({
    "Mean Viewport Hit": "{:.1%}",
    "Minimum Viewport Hit": "{:.1%}",
    "Avg Bandwidth (Mbps)": "{:.1f}",
}))

plt.figure(figsize=(8, 5))
plt.plot(
    guard_table["Avg Bandwidth (Mbps)"],
    guard_table["Mean Viewport Hit"] * 100,
    marker="o"
)
for _, row in guard_table.iterrows():
    plt.text(
        row["Avg Bandwidth (Mbps)"] + 0.2,
        row["Mean Viewport Hit"] * 100,
        row["Guard Band"]
    )
plt.xlabel("Average Bandwidth (Mbps)")
plt.ylabel("Mean Viewport Hit Ratio (%)")
plt.title("Guard Band Trade-off: Bandwidth vs Robustness")
plt.grid(alpha=0.2)
plt.show()


# 第八幕：Head Pose Prediction

Guard Band 是“多传一点来保险”。

另一条路线是：

> **预测用户在网络延迟之后会看哪里。**

这里不需要 GPU，先做一个最简单的 Linear Velocity Predictor：

\[
\hat{\theta}(t+\Delta)
=
\theta(t)
+
v(t)\Delta
\]

服务器拿到的是 200 ms 前的 pose，但可以根据最近两个历史 pose 外推当前方向。

比较：

- No Prediction
- Linear Prediction
- Linear Prediction + 20° Guard


In [9]:
def predict_current_yaw(index, latency_ms):
    lag = int(round(latency_ms / 1000 * FPS))
    j = max(0, index - lag)

    if j == 0:
        return float(yaw_trace[j])

    velocity_per_frame = angular_diff(
        yaw_trace[j],
        yaw_trace[j - 1]
    )

    return float(yaw_trace[j] + velocity_per_frame * lag)

prediction_rows = []
prediction_curves = {}

strategies = [
    ("No Prediction", False, 0),
    ("Linear Prediction", True, 0),
    ("Prediction + 20° Guard", True, 20),
]

for name, use_prediction, guard in strategies:
    ratios = []
    errors = []
    bandwidths = []

    for i in range(N_FRAMES):
        current = tiles_for_pose(yaw_trace[i], pitch_trace[i], 0)

        if use_prediction:
            estimated_yaw = predict_current_yaw(i, LATENCY_MS)
        else:
            j = delayed_pose(i, LATENCY_MS)
            estimated_yaw = float(yaw_trace[j])

        delivered = tiles_for_pose(
            estimated_yaw,
            0,
            guard
        )

        ratios.append(hit_ratio(current, delivered))
        errors.append(abs(angular_diff(estimated_yaw, yaw_trace[i])))
        bandwidths.append(bandwidth_for_hq_tiles(len(delivered)))

    prediction_curves[name] = np.array(ratios)

    prediction_rows.append({
        "Strategy": name,
        "Mean Angular Error (deg)": np.mean(errors),
        "Mean Viewport Hit": np.mean(ratios),
        "Minimum Viewport Hit": np.min(ratios),
        "Avg Bandwidth (Mbps)": np.mean(bandwidths),
    })

prediction_table = pd.DataFrame(prediction_rows)

display(prediction_table.style.format({
    "Mean Angular Error (deg)": "{:.1f}",
    "Mean Viewport Hit": "{:.1%}",
    "Minimum Viewport Hit": "{:.1%}",
    "Avg Bandwidth (Mbps)": "{:.1f}",
}))

plt.figure(figsize=(11, 5))
for name, ratios in prediction_curves.items():
    plt.plot(TIMES, ratios*100, marker="o", label=name)
plt.xlabel("Time (s)")
plt.ylabel("Viewport Hit Ratio (%)")
plt.title("Head Pose Prediction Helps During Network Latency")
plt.ylim(0, 105)
plt.grid(alpha=0.2)
plt.legend()
plt.show()


# 第九幕：把 Tile Selection 变成真正可看的 Viewport Quality

为了把抽象 Hit Ratio 变成视觉效果：

- HQ tile：保留原图
- LQ tile：明显降采样再放大

然后使用同一个 Perspective Viewport extractor。

这样，用户快速转头时会真正看到：

### Viewport Only
高质量区域可能还停在旧方向。

### Prediction + Guard
当前 Viewport 中更多区域已经是高质量。


In [10]:
def make_quality_panorama(panorama, hq_tiles):
    output = panorama.copy()

    tile_h_px = ERP_H // TILE_ROWS
    tile_w_px = ERP_W // TILE_COLS

    for r in range(TILE_ROWS):
        for c in range(TILE_COLS):
            if (r, c) in hq_tiles:
                continue

            y0, y1 = r*tile_h_px, (r+1)*tile_h_px
            x0, x1 = c*tile_w_px, (c+1)*tile_w_px

            tile = panorama[y0:y1, x0:x1]
            low = resize(
                tile,
                (max(6, tile.shape[0]//6), max(8, tile.shape[1]//6)),
                anti_aliasing=True,
                preserve_range=True,
            )
            low = resize(
                low,
                tile.shape,
                order=1,
                anti_aliasing=False,
                preserve_range=True,
            )

            output[y0:y1, x0:x1] = low

    return np.clip(output, 0, 1)

# Pick a fast-turn moment.
idx = 23

current_yaw = float(yaw_trace[idx])
j = delayed_pose(idx, LATENCY_MS)

vp_only_tiles = tiles_for_pose(float(yaw_trace[j]), 0, 0)
pred_guard_tiles = tiles_for_pose(
    predict_current_yaw(idx, LATENCY_MS),
    0,
    20
)

vp_only_panorama = make_quality_panorama(erp, vp_only_tiles)
pred_guard_panorama = make_quality_panorama(erp, pred_guard_tiles)

clean_view = extract_viewport(erp, current_yaw)
vp_only_view = extract_viewport(vp_only_panorama, current_yaw)
pred_guard_view = extract_viewport(pred_guard_panorama, current_yaw)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(clean_view)
axes[0].set_title("Ideal HQ Viewport")
axes[1].imshow(vp_only_view)
axes[1].set_title("Viewport Only\\n200 ms stale pose")
axes[2].imshow(pred_guard_view)
axes[2].set_title("Prediction + 20° Guard")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Fast Head Turn at t={TIMES[idx]:.1f}s | current yaw={current_yaw:.0f}°"
)
plt.tight_layout()
plt.show()


# 第十幕：生成 6 秒动态 Viewport Streaming GIF

下面把真正的用户 Viewport 并排播放：

1. **Ideal Full-Sphere HQ**
2. **Viewport Only / 200 ms stale pose**
3. **Prediction + 20° Guard**

观察 2.0–2.5 s 和 4.3–4.9 s 的快速转头阶段。

这就是整个 Demo7 最重要的动态展示。


In [11]:
gif_path = Path("/tmp/demo7_viewport_streaming.gif")

gif_frames = []

for i in range(N_FRAMES):
    current_yaw = float(yaw_trace[i])

    # Ideal full-sphere HQ
    ideal_view = extract_viewport(erp, current_yaw)

    # Viewport only, stale pose
    j = delayed_pose(i, LATENCY_MS)
    vp_tiles = tiles_for_pose(float(yaw_trace[j]), 0, 0)
    vp_panorama = make_quality_panorama(erp, vp_tiles)
    vp_view = extract_viewport(vp_panorama, current_yaw)

    # Prediction + guard
    predicted_yaw = predict_current_yaw(i, LATENCY_MS)
    pg_tiles = tiles_for_pose(predicted_yaw, 0, 20)
    pg_panorama = make_quality_panorama(erp, pg_tiles)
    pg_view = extract_viewport(pg_panorama, current_yaw)

    views = [
        ("Ideal HQ", ideal_view),
        ("Viewport Only", vp_view),
        ("Prediction + Guard", pg_view),
    ]

    canvas = Image.new("RGB", (VIEW_W*3, VIEW_H), "black")

    for col, (label, view) in enumerate(views):
        pil = Image.fromarray(np.uint8(np.clip(view*255, 0, 255)))
        draw = ImageDraw.Draw(pil)
        draw.rectangle((0, 0, VIEW_W, 24), fill=(0, 0, 0))
        draw.text(
            (8, 5),
            f"{label} | t={TIMES[i]:.1f}s yaw={current_yaw:.0f}°",
            fill=(255, 255, 255),
        )
        canvas.paste(pil, (col*VIEW_W, 0))

    gif_frames.append(canvas)

gif_frames[0].save(
    gif_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=int(1000/FPS),
    loop=0,
    optimize=True,
)

print(
    f"GIF created: {gif_path} | "
    f"size={gif_path.stat().st_size/1024:.1f} KiB"
)

display(IPImage(filename=str(gif_path)))


# 第十一幕：Viewport Streaming ≠ Foveated Streaming

这两个概念经常被混在一起。

### Viewport-Dependent Streaming

问题是：

> **整个球面中，用户当前看哪个方向？**

粒度：

**Sphere → Viewport**

### Foveated Streaming

进一步问：

> **当前 Viewport 中，用户眼睛具体注视哪里？**

粒度：

**Viewport → Fovea / Peripheral Vision**

因此可以理解为：

**Full Sphere → Viewport → Fovea**

空间分配越来越精细。


# 从课堂 Demo 到现实系统

### MPEG OMAF

MPEG-I OMAF（Omnidirectional Media Format）是面向 omnidirectional media 的标准体系。OMAF 第二版引入了根据动态 viewpoint / viewport 组合和呈现内容的能力，使系统能够更有效地只交付与用户当前视野相关的内容。

https://www.mpeg.org/standards/MPEG-I/2/

### ITU-T：Viewport-Dependent VR Streaming

ITU-T H.705.2 将 viewport-dependent VR live streaming 作为 ultra-low-latency streaming 场景之一：内容可以按 tiles 存储，客户端根据 viewport orientation 请求对应 tiles，从而避免下载全部 VR media content。

https://www.itu.int/rec/T-REC-H.705.2-202309-I/en

### Apple Vision Pro：Foveated Streaming

visionOS 26.4 引入 Foveated Streaming framework，可以从本地或云端 streaming endpoint 向 Apple Vision Pro 传输高质量沉浸式内容，并根据用户大致注视区域优先提供高质量内容。

https://developer.apple.com/visionos/whats-new/

这些真实系统共同说明：

> **沉浸式媒体传输正在从“整帧同质量”走向“基于用户空间注意位置的动态质量分配”。**


## 可选 GPU 扩展：Tiny GRU Head Pose Predictor

主实验故意使用 Linear Prediction，因为：

- CPU 即可运行；
- 原理透明；
- 足以展示 Prediction 为什么能改善 Viewport Hit。

如果后续增加 AI 扩展，可以：

**Past N yaw / pitch samples → Tiny GRU → future pose**

比较：

- No Prediction
- Linear Prediction
- GRU Prediction

指标：

- Angular Error
- Viewport Hit Ratio
- Bandwidth

这一扩展才是 Demo7 中真正合理使用 GPU 的位置。


# 七个 Demo 的完整课程链

### Demo1
**如何识别？**  
Rule → Machine Learning

### Demo2
**如何决策？**  
Rule → Reinforcement Learning / ABR

### Demo3
**如何评价视觉质量？**  
PSNR → SSIM → Perceptual Quality

### Demo4
**如何在有限比特下保住质量？**  
Rate → Distortion → Learned Compression

### Demo5
**是否必须传完整信号？**  
Signal-Oriented → Task-Oriented Communication

### Demo6
**网络变差以后，播放器发生什么？**  
Network KPI → Buffer → Playback QoE

### Demo7
**用户下一秒会看哪里？**  
Viewport → Prediction → Foveated Streaming

---

> **Future Media & Internet 的核心，不只是“把更多数据送过去”，而是理解内容、网络、用户和任务，然后把正确的信息，在正确的时间，送到正确的位置。**


## 实验局限性与思考

### 本实验有意做了哪些简化？

1. ERP 是由四张自然图像拼接形成，不是真实 360° 摄像机内容；
2. Tile Grid 只有 8×4；
3. HQ/LQ bitrate 是教学模型；
4. 没有真实 DASH/OMAF bitstream；
5. 网络延迟只体现为 pose/request staleness；
6. Linear predictor 不处理复杂人体运动；
7. 没有 eye tracking；
8. 没有测量 motion-to-photon latency 或 cybersickness；
9. 动态 GIF 用于教学，不是真实 VR renderer。

### 思考题

1. 为什么同样 200 ms latency，在静止观看和快速转头时影响完全不同？
2. Guard Band 为什么不能无限增大？
3. 如果 Tile Grid 从 8×4 提升到 16×8，会带来什么收益和开销？
4. 为什么 Viewport Hit Ratio 比全局 PSNR 更适合本实验？
5. Head Pose Predictor 预测得太远会发生什么？
6. Viewport Streaming 与 Foveated Streaming 的优化目标有什么区别？
7. 怎样把 Demo2 的 ABR 与 Demo7 的 Tile Selection 联合起来？
8. 如果 eye tracking 数据不能离开设备，系统如何同时完成 foveation 与隐私保护？
9. 如何把这个 Demo 扩展成真实 MPEG OMAF / DASH 360° streaming experiment？


---

← [实验六：网络损伤对高清视频的影响](https://www.kaggle.com/code/guopingtan/fmi-demo-6-network-impairments-on-hd-video)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
